# 🖥️ Laptop Price Prediction
## Step 4: Feature Engineering
**Student:** Shivakumar | B.Tech CSE | VEMU Institute of Technology

---
## 📦 0. Import Libraries & Load Cleaned Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
plt.rcParams.update({'figure.dpi': 120, 'axes.titlesize': 12})

# Load cleaned dataset (upload laptop_price_cleaned.csv first)
df = pd.read_csv('laptop_price_cleaned.csv')
df_before = df.copy()

print(f"✅ Cleaned dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Columns: {list(df.columns)}")
df.head()


---
## 📋 1. Before Feature Engineering — Snapshot

In [ ]:
print("=" * 55)
print("    BEFORE FEATURE ENGINEERING")
print("=" * 55)
print(f"  Shape       : {df.shape}")
print(f"  Numeric cols: {df.select_dtypes(include='number').shape[1]}")
print(f"  Object cols : {df.select_dtypes(include='object').shape[1]}")
print("=" * 55)


---
## 🔢 Feature 1: Log Transformation of Target — `log_price`

In [ ]:
# Log transform on target to handle right skew + outlier compression
df['log_price'] = np.log1p(df['Price_euros'])

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['Price_euros'], bins=40, color='#2E5D9E', edgecolor='white', alpha=0.85)
axes[0].set_title('Original Price Distribution', fontweight='bold')
axes[0].set_xlabel('Price (€)')
axes[0].axvline(df['Price_euros'].mean(), color='red', linestyle='--', label=f"Mean: €{df['Price_euros'].mean():.0f}")
axes[0].legend()

axes[1].hist(df['log_price'], bins=40, color='#27AE60', edgecolor='white', alpha=0.85)
axes[1].set_title('log(Price+1) — More Normal Distribution', fontweight='bold')
axes[1].set_xlabel('log(Price + 1)')
axes[1].axvline(df['log_price'].mean(), color='red', linestyle='--', label=f"Mean: {df['log_price'].mean():.2f}")
axes[1].legend()

plt.suptitle('FE 1: Log Transformation of Target Variable', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_01_log_price.png', bbox_inches='tight')
plt.show()

print(f"✅ log_price created")
print(f"  Original skewness : {df['Price_euros'].skew():.3f}")
print(f"  Log skewness      : {df['log_price'].skew():.3f}  (closer to 0 = more normal)")

print("""
📌 Why: Price_euros is right-skewed (skew > 1). 
   Log transform compresses extreme values, makes distribution 
   closer to normal — improves linear model performance significantly.
   np.log1p used (handles 0 safely). Inverse: np.expm1 to recover €.
""")


---
## 🖥️ Feature 2: PPI — Pixels Per Inch (Display Quality Score)

In [ ]:
# PPI already created in cleaning — verify and visualize
print(f"ppi column exists: {'ppi' in df.columns}")
print(f"PPI range: {df['ppi'].min():.1f} — {df['ppi'].max():.1f}")
print(f"PPI mean : {df['ppi'].mean():.1f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['ppi'], bins=35, color='#8E44AD', edgecolor='white', alpha=0.85)
axes[0].set_title('PPI Distribution', fontweight='bold')
axes[0].set_xlabel('Pixels Per Inch')
axes[0].set_ylabel('Count')

axes[1].scatter(df['ppi'], df['Price_euros'], alpha=0.35, color='#8E44AD', s=18)
z = np.polyfit(df['ppi'], df['Price_euros'], 1)
p = np.poly1d(z)
x_line = np.linspace(df['ppi'].min(), df['ppi'].max(), 100)
axes[1].plot(x_line, p(x_line), 'r--', linewidth=2, label='Trend')
axes[1].set_title('PPI vs Price', fontweight='bold')
axes[1].set_xlabel('PPI')
axes[1].set_ylabel('Price (€)')
axes[1].legend()

plt.suptitle('FE 2: PPI — Display Sharpness Feature', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_02_ppi.png', bbox_inches='tight')
plt.show()

print("""
📌 Why: Screen resolution alone is less meaningful than resolution 
   relative to screen size. A 1920×1080 screen on 13" is much sharper 
   than on 17". PPI captures this — higher PPI = premium display = higher price.
   Formula: PPI = sqrt(width² + height²) / screen_inches
""")


---
## 📊 Feature 3: Binning — `ram_tier`

In [ ]:
# Bin RAM into tiers
def ram_tier(gb):
    if gb <= 4:
        return 'Low (≤4GB)'
    elif gb <= 8:
        return 'Mid (8GB)'
    elif gb <= 16:
        return 'High (16GB)'
    else:
        return 'Ultra (32GB+)'

df['ram_tier'] = df['Ram_GB'].apply(ram_tier)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = df['ram_tier'].value_counts()
order = ['Low (≤4GB)', 'Mid (8GB)', 'High (16GB)', 'Ultra (32GB+)']
vc = vc.reindex([o for o in order if o in vc.index])
colors = ['#E74C3C','#F39C12','#27AE60','#2E5D9E'][:len(vc)]

axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('RAM Tier Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=9, fontweight='bold')

order_plot = [o for o in order if o in df['ram_tier'].unique()]
sns.boxplot(data=df, x='ram_tier', y='Price_euros', order=order_plot,
    ax=axes[1], palette=['#E74C3C','#F39C12','#27AE60','#2E5D9E'][:len(order_plot)])
axes[1].set_title('Price by RAM Tier', fontweight='bold')
axes[1].set_xlabel('RAM Tier')
axes[1].set_ylabel('Price (€)')

plt.suptitle('FE 3: RAM Binning into Tiers', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_03_ram_tier.png', bbox_inches='tight')
plt.show()

print(f"✅ ram_tier created")
print(df['ram_tier'].value_counts().to_string())

print("""
📌 Why: Raw RAM values (2,4,6,8,12,16,32,64 GB) are not uniformly spaced 
   in terms of price impact. Binning groups them into meaningful market tiers
   (entry / mid / high / ultra) which aligns with how consumers perceive value.
   Reduces noise from outlier RAM values.
""")


---
## 💾 Feature 4: Binning — `storage_tier`

In [ ]:
def storage_tier(gb):
    if gb <= 128:
        return 'Low (≤128GB)'
    elif gb <= 256:
        return 'Mid (256GB)'
    elif gb <= 512:
        return 'High (512GB)'
    else:
        return 'Ultra (1TB+)'

df['storage_tier'] = df['storage_gb'].apply(storage_tier)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

order = ['Low (≤128GB)', 'Mid (256GB)', 'High (512GB)', 'Ultra (1TB+)']
vc = df['storage_tier'].value_counts().reindex([o for o in order if o in df['storage_tier'].unique()])
colors = ['#E74C3C','#F39C12','#27AE60','#2E5D9E'][:len(vc)]

axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('Storage Tier Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
axes[0].tick_params(axis='x', rotation=15)
for i, v in enumerate(vc.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=9, fontweight='bold')

order_plot = [o for o in order if o in df['storage_tier'].unique()]
sns.boxplot(data=df, x='storage_tier', y='Price_euros', order=order_plot,
    ax=axes[1], palette=['#E74C3C','#F39C12','#27AE60','#2E5D9E'][:len(order_plot)])
axes[1].set_title('Price by Storage Tier', fontweight='bold')
axes[1].set_xlabel('Storage Tier')
axes[1].set_ylabel('Price (€)')
axes[1].tick_params(axis='x', rotation=15)

plt.suptitle('FE 4: Storage Binning into Tiers', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_04_storage_tier.png', bbox_inches='tight')
plt.show()

print(f"✅ storage_tier created")
print(df['storage_tier'].value_counts().to_string())

print("""
📌 Why: Raw storage GB spans 32GB to 2000GB+ with non-linear price impact.
   Binning into market-standard tiers (128 / 256 / 512 / 1TB+) captures 
   the actual consumer perception of storage categories better than raw numbers.
""")


---
## ⚙️ Feature 5: Interaction — `cpu_perf_score` (CPU Brand × GHz)

In [ ]:
# Encode CPU brand as performance multiplier
cpu_brand_score = {
    'Intel': 1.0,
    'AMD'  : 0.9,
    'Apple': 1.2,
    'Samsung': 0.7,
}
df['cpu_brand_score'] = df['cpu_brand'].map(cpu_brand_score).fillna(0.8)

# Interaction: brand weight × clock speed
df['cpu_perf_score'] = (df['cpu_brand_score'] * df['cpu_ghz']).round(3)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(df['cpu_perf_score'], bins=30, color='#C55A11', edgecolor='white', alpha=0.85)
axes[0].set_title('CPU Performance Score Distribution', fontweight='bold')
axes[0].set_xlabel('cpu_perf_score')

axes[1].scatter(df['cpu_perf_score'], df['Price_euros'], alpha=0.35, color='#C55A11', s=18)
z = np.polyfit(df['cpu_perf_score'], df['Price_euros'], 1)
p_line = np.poly1d(z)
x_line = np.linspace(df['cpu_perf_score'].min(), df['cpu_perf_score'].max(), 100)
axes[1].plot(x_line, p_line(x_line), 'r--', linewidth=2)
axes[1].set_title('CPU Perf Score vs Price', fontweight='bold')
axes[1].set_xlabel('cpu_perf_score')
axes[1].set_ylabel('Price (€)')

plt.suptitle('FE 5: CPU Performance Score (Interaction Feature)', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_05_cpu_perf.png', bbox_inches='tight')
plt.show()

print(f"✅ cpu_perf_score created")
print(f"Range: {df['cpu_perf_score'].min():.2f} — {df['cpu_perf_score'].max():.2f}")

print("""
📌 Why: GHz alone doesn't capture CPU quality — a 2.0GHz Apple M1 
   outperforms a 2.0GHz Intel Celeron. This interaction feature multiplies 
   brand tier weight × clock speed to create a single CPU performance proxy.
   Captures cross-feature relationship between brand quality and speed.
""")


---
## 🏆 Feature 6: Feature Creation — `is_premium`

In [ ]:
# Premium laptop flag — combines multiple signals
premium_brands  = ['Apple', 'Microsoft', 'Razer', 'LG']
premium_types   = ['Gaming', 'Workstation', 'Ultrabook']

df['is_premium'] = (
    (df['Company'].isin(premium_brands)) |
    (df['TypeName'].isin(premium_types)) |
    (df['Ram_GB'] >= 16) |
    (df['storage_type'] == 'SSD') & (df['storage_gb'] >= 512)
).astype(int)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

vc = df['is_premium'].value_counts()
axes[0].bar(['Standard (0)', 'Premium (1)'], vc.values,
    color=['#2E5D9E', '#C55A11'], edgecolor='white', alpha=0.9)
axes[0].set_title('Premium vs Standard Count', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=10, fontweight='bold')

sns.boxplot(data=df, x='is_premium', y='Price_euros', ax=axes[1],
    palette=['#2E5D9E', '#C55A11'])
axes[1].set_title('Price: Standard vs Premium', fontweight='bold')
axes[1].set_xticklabels(['Standard', 'Premium'])
axes[1].set_ylabel('Price (€)')

plt.suptitle('FE 6: is_premium — Composite Premium Flag', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_06_premium.png', bbox_inches='tight')
plt.show()

print(f"✅ is_premium created")
print(f"Premium laptops : {df['is_premium'].sum()} ({df['is_premium'].mean()*100:.1f}%)")
print(f"Standard laptops: {(df['is_premium']==0).sum()}")

print("""
📌 Why: No single feature captures 'premium-ness'. This composite binary flag 
   combines brand prestige, laptop type, RAM capacity, and SSD storage to 
   create a strong price signal. Expected to be one of the most important features.
""")


---
## ⚖️ Feature 7: Binning — `weight_category`

In [ ]:
def weight_cat(kg):
    if kg < 1.5:
        return 'Ultralight'
    elif kg < 2.0:
        return 'Light'
    elif kg < 2.5:
        return 'Standard'
    else:
        return 'Heavy'

df['weight_category'] = df['Weight_kg'].apply(weight_cat)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

order = ['Ultralight', 'Light', 'Standard', 'Heavy']
vc = df['weight_category'].value_counts().reindex([o for o in order if o in df['weight_category'].unique()])
colors = ['#27AE60','#2E5D9E','#F39C12','#E74C3C'][:len(vc)]

axes[0].bar(vc.index, vc.values, color=colors, edgecolor='white', alpha=0.9)
axes[0].set_title('Weight Category Distribution', fontweight='bold')
axes[0].set_ylabel('Count')
for i, v in enumerate(vc.values):
    axes[0].text(i, v+5, str(v), ha='center', fontsize=9, fontweight='bold')

order_plot = [o for o in order if o in df['weight_category'].unique()]
sns.boxplot(data=df, x='weight_category', y='Price_euros', order=order_plot,
    ax=axes[1], palette=['#27AE60','#2E5D9E','#F39C12','#E74C3C'][:len(order_plot)])
axes[1].set_title('Price by Weight Category', fontweight='bold')
axes[1].set_ylabel('Price (€)')

plt.suptitle('FE 7: Weight Category Binning', fontweight='bold')
plt.tight_layout()
plt.savefig('fe_07_weight_cat.png', bbox_inches='tight')
plt.show()

print(f"✅ weight_category created")
print(df['weight_category'].value_counts().to_string())

print("""
📌 Why: Raw weight in kg has a non-linear relationship with price.
   Ultralight laptops (thin & light ultrabooks) tend to cost more 
   due to premium materials. Binning into 4 weight classes captures 
   this market segment distinction better than the raw float value.
""")


---
## 🔗 8. New Features vs Price — Correlation Check

In [ ]:
# Numerical features correlation with price
num_features = ['Ram_GB', 'Weight_kg', 'Inches', 'ppi', 'cpu_ghz',
                'storage_gb', 'cpu_perf_score', 'is_touchscreen',
                'is_ips', 'is_premium', 'log_price', 'Price_euros']

corr_df = df[num_features].corr()['Price_euros'].drop('Price_euros').sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(9, 6))
colors = ['#27AE60' if v > 0 else '#E74C3C' for v in corr_df.values]
bars = ax.barh(corr_df.index, corr_df.values, color=colors, edgecolor='white', alpha=0.85)
ax.axvline(0, color='black', linewidth=0.8)
ax.set_title('Feature Correlation with Price_euros\n(After Feature Engineering)', fontweight='bold', fontsize=13)
ax.set_xlabel('Pearson Correlation Coefficient')
for bar, val in zip(bars, corr_df.values):
    ax.text(val + (0.005 if val >= 0 else -0.005), bar.get_y() + bar.get_height()/2,
        f'{val:.3f}', va='center', ha='left' if val >= 0 else 'right', fontsize=9, fontweight='bold')
plt.tight_layout()
plt.savefig('fe_08_correlation.png', bbox_inches='tight')
plt.show()

print("\nCorrelation with Price_euros (sorted):")
print(corr_df.round(3).to_string())


---
## 💾 9. Save Feature Engineered Dataset

In [ ]:
# Drop intermediate helper column
df.drop(columns=['cpu_brand_score'], inplace=True, errors='ignore')

df.to_csv('laptop_price_featured.csv', index=False)
print(f"✅ Saved: laptop_price_featured.csv")
print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")

print("\nAll columns:")
for col in df.columns:
    print(f"  • {col:<25} dtype: {df[col].dtype}")


---
## ✅ 10. Feature Engineering Summary

In [ ]:
print("=" * 65)
print("       FEATURE ENGINEERING SUMMARY")
print("=" * 65)

features = [
    ("1", "log_price",        "Log Transform",   "Normalize right-skewed target variable"),
    ("2", "ppi",              "Feature Creation","Display quality: pixels per inch"),
    ("3", "ram_tier",         "Binning",         "RAM grouped into 4 market tiers"),
    ("4", "storage_tier",     "Binning",         "Storage grouped into 4 capacity tiers"),
    ("5", "cpu_perf_score",   "Interaction",     "cpu_brand_weight × cpu_ghz"),
    ("6", "is_premium",       "Feature Creation","Composite premium laptop flag"),
    ("7", "weight_category",  "Binning",         "Weight grouped into 4 categories"),
]

print(f"  {'#':<3} {'Feature':<20} {'Technique':<18} {'Reason'}")
print(f"  {'-'*62}")
for num, feat, tech, reason in features:
    print(f"  {num:<3} {feat:<20} {tech:<18} {reason}")

print("=" * 65)
print(f"  Original features : {df_before.shape[1]}")
print(f"  New features added: {df.shape[1] - df_before.shape[1]}")
print(f"  Final feature count: {df.shape[1]}")
print("=" * 65)
print("\n📌 Next Step → Data Preprocessing (Step 5)")

# Download
from google.colab import files
files.download('laptop_price_featured.csv')
print("\n✅ laptop_price_featured.csv downloaded!")
